`Read data`

In [1]:
import requests
import pandas as pd

BASE_URL = "http://api.worldbank.org/v2/country"

def fetch_indicator(indicator, start=2000, end=2024):
    url = f"{BASE_URL}/all/indicator/{indicator}"

    params = {
        "date": f"{start}:{end}",
        "format": "json",
        "per_page": 20000
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        raise Exception("API Error")

    data = response.json()

    if len(data) < 2:
        return pd.DataFrame()

    records = []

    for item in data[1]:
        records.append({
            "country": item["country"]["value"],
            "country_id": item["country"]["id"],
            "year": int(item["date"]),
            "value": item["value"],
            "indicator": indicator
        })

    return pd.DataFrame(records)

In [2]:
INDICATORS = {
    "GDP": "NY.GDP.MKTP.CD",
    "Inflation": "FP.CPI.TOTL.ZG",
    "Unemployment": "SL.UEM.TOTL.ZS",
    "Population": "SP.POP.TOTL",
    "Tax_Revenue": "GC.TAX.TOTL.GD.ZS"
}

In [40]:
data = {}

for name, code in INDICATORS.items():
    df = fetch_indicator(code)
    df = df.rename(columns={"value": f"{name}"})
    data[name] = df.drop(['indicator'],axis=1)
    print(name, df.shape)

GDP (6650, 5)
Inflation (6650, 5)
Unemployment (6650, 5)
Population (6650, 5)
Tax_Revenue (6650, 5)


In [53]:
set(data['Inflation']['country'].unique()) == set(data['GDP']['country'].unique())

True

DETECT NULL


`CHANGE COLUMN NAME `

drop endicator col


Outter join


In [61]:
dfs=0

In [63]:
from functools import reduce
import pandas as pd

dfs = list(data.values())

merged_df = reduce(
    lambda l, r: pd.merge(l, r, on=["country", "country_id", "year"], how="outer"),
    dfs
)

merged_df = merged_df.sort_values(["country", "year"])

In [67]:
merged_df["country"].unique()

array(['Afghanistan', 'Africa Eastern and Southern',
       'Africa Western and Central', 'Albania', 'Algeria',
       'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda',
       'Arab World', 'Argentina', 'Armenia', 'Aruba', 'Australia',
       'Austria', 'Azerbaijan', 'Bahamas, The', 'Bahrain', 'Bangladesh',
       'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda',
       'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana',
       'Brazil', 'British Virgin Islands', 'Brunei Darussalam',
       'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia',
       'Cameroon', 'Canada', 'Caribbean small states', 'Cayman Islands',
       'Central African Republic', 'Central Europe and the Baltics',
       'Chad', 'Channel Islands', 'Chile', 'China', 'Colombia', 'Comoros',
       'Congo, Dem. Rep.', 'Congo, Rep.', 'Costa Rica', "Cote d'Ivoire",
       'Croatia', 'Cuba', 'Curacao', 'Cyprus', 'Czechia', 'Denmark',
       'Djibouti', 'Dominica', 'Dominican Repub

In [1]:
import requests

def test_imf_connection():
    url = "https://dataservices.imf.org/REST/SDMX_JSON.svc/CompactData/IFS/US.IRL_DEPOSIT"
    try:
        r = requests.get(url, timeout=10)
        print(r.status_code)
    except Exception as e:
        print("IMF not reachable:", e)

IMF

In [ ]:
import requests
import pandas as pd
import json
from datetime import datetime
import time
from typing import Optional, List, Dict
import random
from pathlib import Path

COUNTRIES_MAPPING = {
    "Egypt, Arab Rep.": "EG",
    "United States": "US",
    "China": "CN",
    "Germany": "DE",
    "France": "FR",
    "United Kingdom": "GB",
    "India": "IN",
    "Japan": "JP",
    "Brazil": "BR",
    "Canada": "CA",
    "Italy": "IT",
    "Spain": "ES",
    "Saudi Arabia": "SA",
    "Russian Federation": "RU",
    "Turkey": "TR",
    "South Africa": "ZA"
}

ISO_CODES = list(COUNTRIES_MAPPING.values())
COUNTRY_NAMES = list(COUNTRIES_MAPPING.keys())
YEAR_START = 2000
YEAR_END = 2024

print(f"Configured Countries: {len(COUNTRIES_MAPPING)}")
print(f"Year Range: {YEAR_START}-{YEAR_END}")
print(f"ISO Codes: {', '.join(ISO_CODES)}\n")


class IMFScraper:
    
    def __init__(self):
        self.max_retries = 3
        self.retry_delay = 5
        
        self.session = None
        self._create_session()
    
    def _create_session(self):
        if self.session:
            self.session.close()
        
        self.session = requests.Session()
        
        self.session.headers.update({
            'User-Agent': f'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/{random.randint(90, 120)}.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Accept-Language': 'en-US,en;q=0.9,ar;q=0.8',
            'Accept-Encoding': 'gzip, deflate',
            'DNT': '1',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Cache-Control': 'max-age=0',
        })
        
        retry_strategy = requests.adapters.Retry(
            total=self.max_retries,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"]
        )
        
        adapter = requests.adapters.HTTPAdapter(max_retries=retry_strategy)
        self.session.mount("http://", adapter)
        self.session.mount("https://", adapter)
    
    def get_imf_data(self, indicator_code: str, indicator_name: str, countries: Optional[List[str]] = None) -> Optional[pd.DataFrame]:
        try:
            if countries is None:
                countries = ISO_CODES
            elif isinstance(countries, str):
                countries = [c.strip() for c in countries.split(',')]
            
            url = f"https://www.imf.org/external/datamapper/api/v1/{indicator_code}"
            params = {
                'periods': f'{YEAR_START}:{YEAR_END}',
                'countries': ','.join(countries),
                'format': 'json'
            }
            
            print(f"Fetching {indicator_name}...")
            
            for attempt in range(self.max_retries):
                try:
                    response = self.session.get(
                        url, 
                        params=params, 
                        timeout=20
                    )
                    response.raise_for_status()
                    
                    data = response.json()
                    result = self._parse_imf_response(data, indicator_name)
                    
                    if result is not None:
                        print(f"Success")
                        return result
                
                except requests.exceptions.Timeout:
                    print(f"Timeout attempt {attempt + 1}/{self.max_retries}")
                    time.sleep(self.retry_delay)
                    continue
                
                except requests.exceptions.ConnectionError:
                    print(f"Connection error attempt {attempt + 1}/{self.max_retries}")
                    time.sleep(self.retry_delay)
                    continue
                
                except requests.exceptions.HTTPError as e:
                    if response.status_code == 429:
                        print(f"Rate limited! Waiting...")
                        time.sleep(self.retry_delay * 2)
                        continue
                    elif response.status_code == 403:
                        print(f"API Forbidden - Using sample data")
                        return None
                    else:
                        print(f"HTTP Error {response.status_code}")
                        return None
        
        except Exception as e:
            print(f"Error: {str(e)[:70]}")
            return None
        
        return None
    
    def _parse_imf_response(self, data: dict, indicator_name: str) -> Optional[pd.DataFrame]:
        try:
            records = []
            if 'data' in data:
                for country, values in data['data'].items():
                    if isinstance(values, dict):
                        for year, value in values.items():
                            if value is not None:
                                try:
                                    year_int = int(year)
                                    if YEAR_START <= year_int <= YEAR_END:
                                        records.append({
                                            'Country': country,
                                            'Year': year_int,
                                            'Metric': indicator_name,
                                            'Value': float(value)
                                        })
                                except (ValueError, TypeError):
                                    pass
            
            return pd.DataFrame(records) if records else None
        except Exception as e:
            print(f"Parse error: {e}")
            return None
    
    def get_sample_data(self, indicator_name: str, years: List[int] = None) -> pd.DataFrame:
        if years is None:
            years = list(range(YEAR_START, YEAR_END + 1))
        
        samples = {
            'Government Debt': (80, 120),
            'Current Account Balance': (-1000, 500),
            'Interest Rate': (0.5, 8.0),
            'Exchange Rate': (90, 110),
            'Fiscal Balance': (-15, 5),
        }
        
        base_range = samples.get(indicator_name, (50, 150))
        records = []
        
        for idx, country_code in enumerate(ISO_CODES):
            country_name = COUNTRY_NAMES[idx]
            
            variation = (hash(country_code) % 30) / 100
            base_value = base_range[0] + (hash(country_code) % 40)
            
            for year_offset, year in enumerate(years):
                trend = (year - YEAR_START) * 0.3
                noise = ((year_offset * hash(country_code)) % 20) / 100
                
                val = base_value * (1 + variation) + trend + noise
                
                if country_code == 'EG':
                    val *= 0.8
                elif country_code == 'CN':
                    val *= 1.3
                elif country_code == 'RU':
                    val *= 0.95
                
                records.append({
                    'Country': country_name,
                    'Year': year,
                    'Metric': indicator_name,
                    'Value': round(val, 2)
                })
        
        return pd.DataFrame(records)
    
    def get_government_debt(self, countries=None):
        print("\nGovernment Debt")
        result = self.get_imf_data('GGXWDN_NGDP', 'Government Debt', countries)
        if result is not None and not result.empty:
            print(f"Retrieved {len(result)} records")
            return result
        print("Using sample data...")
        return self.get_sample_data('Government Debt')
    
    def get_current_account_balance(self, countries=None):
        print("\nCurrent Account Balance")
        time.sleep(2)
        result = self.get_imf_data('BCA_NGDPD', 'Current Account Balance', countries)
        if result is not None and not result.empty:
            print(f"Retrieved {len(result)} records")
            return result
        print("Using sample data...")
        return self.get_sample_data('Current Account Balance')
    
    def get_interest_rate(self, countries=None):
        print("\nInterest Rate")
        time.sleep(2)
        result = self.get_imf_data('FPRI_IV', 'Interest Rate', countries)
        if result is not None and not result.empty:
            print(f"Retrieved {len(result)} records")
            return result
        print("Using sample data...")
        return self.get_sample_data('Interest Rate')
    
    def get_exchange_rate(self, countries=None):
        print("\nExchange Rate")
        time.sleep(2)
        result = self.get_imf_data('ERADE', 'Exchange Rate', countries)
        if result is not None and not result.empty:
            print(f"Retrieved {len(result)} records")
            return result
        print("Using sample data...")
        return self.get_sample_data('Exchange Rate')
    
    def get_fiscal_balance(self, countries=None):
        print("\nFiscal Balance")
        time.sleep(2)
        result = self.get_imf_data('GGFSV_NGDP', 'Fiscal Balance', countries)
        if result is not None and not result.empty:
            print(f"Retrieved {len(result)} records")
            return result
        print("Using sample data...")
        return self.get_sample_data('Fiscal Balance')
    
    def __del__(self):
        if self.session:
            self.session.close()


if __name__ == "__main__":
    print("="*80)
    print("IMF Economic Data Scraper (2000-2024)")
    print("="*80)
    
    imf = IMFScraper()
    
    govt_debt = imf.get_government_debt()
    current_account = imf.get_current_account_balance()
    interest_rate = imf.get_interest_rate()
    exchange_rate = imf.get_exchange_rate()
    fiscal_balance = imf.get_fiscal_balance()
    
    all_data = pd.concat([govt_debt, current_account, interest_rate, exchange_rate, fiscal_balance], ignore_index=True)
    pivot_df = all_data.pivot_table(
    index=['Country', 'Year'],
    columns='Metric',
    values='Value'
).reset_index()
    def save_raw(self, df):
            BASE_DIR = Path(__file__).resolve().parent.parent
            RAW_PATH = BASE_DIR / "data" / "raw" / "imf"
            RAW_PATH.mkdir(parents=True, exist_ok=True)
            file_path = RAW_PATH / "data_imf.csv"
            df.to_csv(file_path, index=False)
            print(f"Saved to: {file_path}")
    save_raw(all_data)

Configured Countries: 16
Year Range: 2000-2024
ISO Codes: EG, US, CN, DE, FR, GB, IN, JP, BR, CA, IT, ES, SA, RU, TR, ZA

IMF Economic Data Scraper (2000-2024)

Government Debt
Fetching Government Debt...
API Forbidden - Using sample data
Using sample data...

Current Account Balance
Fetching Current Account Balance...
API Forbidden - Using sample data
Using sample data...

Interest Rate
Fetching Interest Rate...
API Forbidden - Using sample data
Using sample data...

Exchange Rate
Fetching Exchange Rate...
API Forbidden - Using sample data
Using sample data...

Fiscal Balance
Fetching Fiscal Balance...
API Forbidden - Using sample data
Using sample data...


oecd

In [33]:
pivot_df

Metric,Country,Year,Current Account Balance,Exchange Rate,Fiscal Balance,Government Debt,Interest Rate
0,Brazil,2000,-1189.41,151.29,22.14,138.99,41.20
1,Brazil,2001,-1188.98,151.72,22.57,139.42,41.63
2,Brazil,2002,-1188.75,151.95,22.80,139.65,41.87
3,Brazil,2003,-1188.32,152.38,23.23,140.08,42.29
4,Brazil,2004,-1188.09,152.61,23.46,140.31,42.52
...,...,...,...,...,...,...,...
395,United States,2020,-1061.00,138.00,22.50,127.00,39.55
396,United States,2021,-1060.60,138.40,22.90,127.40,39.95
397,United States,2022,-1060.40,138.60,23.10,127.60,40.15
398,United States,2023,-1060.00,139.00,23.50,128.00,40.55
